# Лабораторная работа 4

Ноутбук специально простой: здесь только вызовы функций из `src/*.py`, таблицы и графики. Реализация сети, optimizers и обучения лежит в Python-файлах.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

from src.data import make_lab_datasets
from src.experiments import run_experiments

RANDOM_STATE = 467866
OUTPUT_DIR = ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

## 1. Генерация данных

Датасеты создаются строго по условию: `make_moons(400, noise=0.15)` и `make_classification(200, n_features=5, n_redundant=2, n_informative=2, n_clusters_per_class=2)`.

In [ ]:
datasets = make_lab_datasets(RANDOM_STATE)

for key, data in datasets.items():
    print(data["name"])
    print("  full:", data["X"].shape)
    print("  train:", data["X_train"].shape)
    print("  val:", data["X_val"].shape)
    print("  test:", data["X_test"].shape)

## 2. Обучение

`run_experiments` обучает две модели на каждом датасете: BReLU + Sigmoid и перцептрон с одним скрытым слоем. Внутри перебираются `hidden ∈ {8, 16, 32}` и оптимизаторы `SGD / Adam`.

In [ ]:
result = run_experiments(seed=RANDOM_STATE, output_dir=OUTPUT_DIR)

## 3. Итоговые метрики лучших моделей

In [ ]:
best = pd.read_csv(TABLES_DIR / "best_results.csv")
cols = [
    "dataset", "model", "optimizer", "hidden", "lr", "batch_size", "epochs_done",
    "val_accuracy", "val_f1", "test_accuracy", "test_precision", "test_recall", "test_f1"
]
best[cols]

## 4. Сравнение test accuracy и test F1

In [ ]:
for fig in ["metrics_accuracy_comparison.png", "metrics_f1_comparison.png"]:
    display(Image(filename=str(FIGURES_DIR / fig)))

## 5. Визуализация датасетов

In [ ]:
for fig in ["dataset_moons.png", "dataset_classification.png"]:
    display(Image(filename=str(FIGURES_DIR / fig)))

## 6. Кривые обучения

In [ ]:
for fig in sorted(FIGURES_DIR.glob("learning_loss_*.png")):
    display(Image(filename=str(fig)))

for fig in sorted(FIGURES_DIR.glob("learning_accuracy_*.png")):
    display(Image(filename=str(fig)))

## 7. Границы решений

Для пятимерного `make_classification` границы показаны в PCA-проекции: модель обучалась на всех 5 признаках, а график нужен только для визуализации.

In [ ]:
for fig in sorted(FIGURES_DIR.glob("decision_boundary_*.png")):
    display(Image(filename=str(fig)))

## 8. Полный перебор гиперпараметров

In [ ]:
grid = pd.read_csv(TABLES_DIR / "grid_results.csv")
grid[["dataset", "model", "optimizer", "hidden", "lr", "batch_size", "val_accuracy", "val_f1", "test_accuracy", "test_f1"]]